In [29]:
import pandas as pd

# Load dataset
df = pd.read_csv('IPL.csv', low_memory=False)

# Display first 5 rows
df.head()

,Unnamed: 0,match_id,date,match_type,event_name,innings,batting_team,bowling_team,over,ball,...,team_balls,team_wicket,new_batter,power_surge_start,batter_runs,batter_balls,bowler_wicket,batting_partners,next_batter,striker_out
0,141607,335982,4/18/2008,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,...,1,0,NaN,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
1,141608,335982,4/18/2008,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,...,2,0,NaN,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
2,141609,335982,4/18/2008,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,...,2,0,NaN,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
3,141610,335982,4/18/2008,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,...,3,0,NaN,NaN,0,2,0,"('BB McCullum', 'SC Ganguly')",NaN,False
4,141611,335982,4/18/2008,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,...,4,0,NaN,NaN,0,3,0,"('BB McCullum', 'SC Ganguly')",NaN,False


In [18]:
# Check missing values per column
df.isnull().sum()

,0
Unnamed: 0,0
match_id,0
date,0
match_type,0
event_name,0
...,...
batter_balls,0
bowler_wicket,0
batting_partners,0
next_batter,270085


In [19]:
# Fill numeric missing values with column mean (preserves distribution)
num_cols = df.select_dtypes(include='number').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

# Fill missing text values with 'Unknown' so rows aren't lost
# (excluding batting_partners, which holds tuple-like player pair data)
text_cols = [c for c in df.select_dtypes(include='object').columns if c != 'batting_partners']
df[text_cols] = df[text_cols].fillna('Unknown')

# batting_partners and next_batter contain player names / NaN for early balls — fill separately
df['batting_partners'] = df['batting_partners'].fillna('Unknown')
df['next_batter'] = df['next_batter'].fillna('Unknown')

In [20]:
# Remove duplicate rows
df = df.drop_duplicates()

In [21]:
# Strip whitespace and standardize casing for text columns
# (excluding batting_partners, which is tuple-formatted player names, not a simple label)
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

In [14]:
# Convert date column to proper datetime type
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Ensure numeric columns are correct type
df['batter_runs'] = pd.to_numeric(df['batter_runs'], errors='coerce')
df['over'] = pd.to_numeric(df['over'], errors='coerce')
df['ball'] = pd.to_numeric(df['ball'], errors='coerce')

In [24]:
# Rename for clarity/consistency
df = df.rename(columns={
    'batter_runs': 'runs_scored',
    'batter_balls': 'balls_faced',
    'bowler_wicket': 'wicket_taken'
})

In [26]:
# Flag whether a delivery was a boundary (4 or 6 runs)
df['is_boundary'] = df['runs_scored'].apply(lambda x: 1 if x in [4, 6] else 0)

In [28]:
df.to_csv('cleaned_dataset.csv', index=False)